<a href="https://colab.research.google.com/github/vivacitylabs/data-toolkit/blob/master/notebooks/classified_journey_times_bulk_download_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classified Journey Times - Bulk Download Generator


## Generate a csv file of Classified Journey Times data over multiple days

This notebook is a tool to access VivaCity data via the API. It is aimed as an **interim solution** while we're working on new dashboard developments. You can contact customer support if you have any issues (support@vivacitylabs.com) or raise a ticket on the [Customer Help Portal](https://vivacitylabs.atlassian.net/servicedesk/customer/portal/16).

Journey times are measured between **pairs of ANPR-enabled countlines**. Only countlines with ANPR enabled are listed below. A journey is registered when a number plate is captured and deciphered at both the origin and destination countline, and only journeys **under 2 hours** in length are returned. You can download a combined **total** across all classes, or a **per-class** breakdown — pick this with the **Classes** selector in Stage 2.

#### How it works

This notebook will run you through all the necessary steps and will save the output csv file locally or in your Google Drive.

You will simply need to fill in a few details and then hit the run button (▶) next to the code cells.

**What you will need**

- A VivaCity API key
- The ANPR countlines you want origin/destination journey times for

#### Output format

The output file will have one row per origin/destination countline pair, time bucket and class:

| UTC Datetime | Local Datetime | Origin Countline | Destination Countline | Class | Journeys | 25 Percentile | Mean Journey Time | Median Journey Time | 75 Percentile |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| 2025-05-06 16:00:00+00:00 | 2025-05-06 17:00:00+01:00 | projectA / Countline 1 (45821) | projectA / Countline 1 (45823) | car | 1 | 4437 | 4437 | 4437 | 4437 |

All journey time values (25th percentile, mean, median, 75th percentile) are given in **seconds**.

## Stage 1: Getting Started
Let's begin by importing the packages we'll need and creating some useful functions!

Hit the run button (▶) in the top left corner.

In [ ]:
#@title  { vertical-output: true, display-mode: "form" }
#@markdown **Code cell:** Run this to import functions
import requests
import json
import time
import threading
from collections import deque
from datetime import date, datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import pytz
from IPython.display import Markdown, display
from tqdm.auto import tqdm
from ipywidgets import Layout, Box
import ipywidgets as widgets

def printmd(string):
    display(Markdown(string))

def get_date_range(start_date, end_date):
    """Split the requested period into one-day [from, to) windows (UTC)."""
    start_dates = []
    end_dates = []
    start_date = datetime.fromisoformat(start_date)
    end_date = datetime.fromisoformat(end_date)
    while True:
        start_dates.append(start_date.strftime('%Y-%m-%dT%H:%M:%S.000Z'))
        end_dates.append((start_date + timedelta(days=1)).strftime('%Y-%m-%dT%H:%M:%S.000Z'))
        start_date = start_date + timedelta(days=1)
        if start_date > end_date:
            break
    return list(zip(start_dates, end_dates))

class RateLimiter:
    """Thread-safe limiter respecting the API's 300/minute and 7200/hour caps.

    We sit just under the documented limits to leave headroom for retries."""
    def __init__(self, max_per_min=290, max_per_hour=7000):
        self.max_per_min = max_per_min
        self.max_per_hour = max_per_hour
        self.minute = deque()
        self.hour = deque()
        self.lock = threading.Lock()

    def acquire(self):
        while True:
            with self.lock:
                now = time.time()
                while self.minute and now - self.minute[0] > 60:
                    self.minute.popleft()
                while self.hour and now - self.hour[0] > 3600:
                    self.hour.popleft()
                if len(self.minute) < self.max_per_min and len(self.hour) < self.max_per_hour:
                    self.minute.append(now)
                    self.hour.append(now)
                    return
                waits = []
                if len(self.minute) >= self.max_per_min:
                    waits.append(60 - (now - self.minute[0]))
                if len(self.hour) >= self.max_per_hour:
                    waits.append(3600 - (now - self.hour[0]))
                wait = max(0.01, min(waits))
            time.sleep(wait)

print("Done.")

## Stage 2: Data Import

1. First, enter your API key to authenticate.
2. We then fetch hardware metadata and list every **ANPR-enabled** countline you have access to.
3. You select the countlines, date range, timezone, time bucket and (optionally) classes to request.

ℹ️ Journey times are requested for **every ordered origin → destination pair** of the countlines you select, so the number of API requests grows **quadratically** with the number of countlines (and linearly with the number of days). Pick a larger time bucket and a focused set of countlines if you want to keep the download fast.

### Authentication
Enter your VivaCity API key below and hit the run button (▶). If you don't have one, please contact customer support (support@vivacitylabs.com).

In [ ]:
#@title  { run: "auto", vertical-output: true, display-mode: "form" }
#@markdown Insert your API key
api_key = "" #@param {type:"string"}

headers = {}
headers['x-vivacity-api-key'] = api_key

#### Available ANPR Countlines

Run this to retrieve every countline with ANPR enabled from the hardware metadata endpoint. Countlines are listed by **project name, countline name, countline id and sensor number** because countline names alone are not unique.

In [ ]:
#@title  { vertical-output: true, display-mode: "form" }
#@markdown **Code cell:** Run this to retrieve all ANPR-enabled countlines

api_url_base = 'https://api.vivacitylabs.com'

print("Requesting hardware metadata ...")
hardware_request = requests.get(f'{api_url_base}/hardware/metadata', headers=headers, timeout=120)
if hardware_request.status_code == 401:
    print("\n!Error: Can't access the data. Check your API key, or ask customer support if your user "
          "is set up correctly.\n")
hardware = hardware_request.json()

# Walk the metadata hierarchy (hardware -> viewpoint -> countline) and keep ANPR countlines only
records = []
for hardware_id, hw in hardware.items():
    project_name = hw.get("project_name")
    sensor_number = hw.get("sensor_number")
    for viewpoint_id, viewpoint in (hw.get("view_points") or {}).items():
        for countline_id, countline in (viewpoint.get("countlines") or {}).items():
            if countline.get("is_anpr"):
                records.append({
                    "project_name": project_name,
                    "countline_name": countline.get("name"),
                    "countline_id": countline_id,
                    "sensor_number": sensor_number,
                    "hardware_id": hardware_id,
                })

df_anpr = pd.DataFrame.from_dict(records).drop_duplicates(subset="countline_id").reset_index(drop=True)

# Human-readable label for the selection dropdown and for the output file
def _fmt_sensor(s):
    return str(s) if s is not None else "n/a"

df_anpr["display"] = (
    df_anpr["project_name"].fillna("unknown-project") + "  |  "
    + df_anpr["countline_name"].fillna("(no name)") + "  |  id:" + df_anpr["countline_id"]
    + "  |  sensor:" + df_anpr["sensor_number"].apply(_fmt_sensor)
)
df_anpr["output_label"] = (
    df_anpr["project_name"].fillna("unknown-project") + " / "
    + df_anpr["countline_name"].fillna("(no name)") + " (" + df_anpr["countline_id"] + ")"
)
df_anpr = df_anpr.sort_values(["project_name", "countline_name", "countline_id"]).reset_index(drop=True)

print(len(df_anpr), "ANPR countlines available")

#### Select countlines and date range for querying the API

Run the cell below, then make your selections. Journey times will be requested for **all ordered pairs** of the countlines you pick. Ensure the start date is before the end date.

**Classes** controls how the data is broken down:
- Select **`aggregated classes`** for a single combined **total** per time bucket (other class selections are ignored).
- Select one or more specific classes for a per-class breakdown (a `total` row is also returned).
- Leave it empty for **all** classes, disaggregated.

Smaller time buckets limit the query range and greatly increase the data volume — see the note above.

In [ ]:
#@title  { vertical-output: true, display-mode: "form" }
#@markdown **Code cell:** Run this and then make your selections

box_layout = Layout(display='flex', flex_flow='column',
                    align_items='stretch', border=None, width='60%')

start_date_input = widgets.DatePicker(description="Start date", layout=Layout(width='55%'))
end_date_input = widgets.DatePicker(description="End date", layout=Layout(width='55%'))
timezone = widgets.Dropdown(
    options=['Europe/London', "Europe/Berlin", "America/New_York",
             "Australia/Adelaide", "Australia/Brisbane", "Australia/Darwin",
             "Australia/Melbourne", "Australia/Perth", "Australia/Sydney"],
    description="Timezone", layout=Layout(width='55%'))
timebucket_input = widgets.Dropdown(
    options=['15m', '30m', '1h', '4h', '12h', '24h'], value='1h',
    description="Time bucket", layout=Layout(width='55%'))

# "aggregated classes" returns a single combined total; the rest give a per-class breakdown
class_input = widgets.SelectMultiple(
    options=["aggregated classes", "cyclist", "motorbike", "car", "taxi", "van", "minibus",
             "bus", "rigid", "truck", "emergency_car", "emergency_van", "fire_engine"],
    description='Classes', disabled=False,
    layout=Layout(width='55%', height='200px'))
countlines_input = widgets.SelectMultiple(
    options=df_anpr["display"].tolist(),
    description='Countlines', disabled=False,
    layout=Layout(width='auto', height='300px'))

items = [start_date_input, end_date_input, timezone, timebucket_input, class_input, countlines_input]
box = Box(children=items, layout=box_layout)
printmd("**Select date period, time bucket and countlines**")
printmd("Hold  `Ctrl`  (or  `Cmd`)  to select multiple classes or countlines")
print("")
box

Run the cell below to set the API request parameters and check your selection. It also estimates how many requests are needed and roughly how long the download will take.

In [ ]:
#@title  { vertical-output: true, display-mode: "form" }
#@markdown **Code cell:** Run this and check your selection again.

# Resolve selected countlines back to ids and labels
selected = df_anpr[df_anpr["display"].isin(countlines_input.value)]
selected_ids = selected["countline_id"].tolist()
label_map = dict(zip(selected["countline_id"], selected["output_label"]))

# Ordered origin -> destination pairs (exclude self-pairs)
pairs = [(o, d) for o in selected_ids for d in selected_ids if o != d]

time_bucket = timebucket_input.value

# Class handling: "aggregated classes" => combined total only; otherwise per-class breakdown
selected_classes = list(class_input.value)
aggregated = "aggregated classes" in selected_classes
request_classes = [c for c in selected_classes if c != "aggregated classes"]
if aggregated:
    classes_summary = "aggregated (total only)"
elif request_classes:
    classes_summary = ", ".join(request_classes) + " (disaggregated)"
else:
    classes_summary = "all (disaggregated)"

if start_date_input.value is None or end_date_input.value is None:
    print("Please select both a start and end date.")
elif start_date_input.value > end_date_input.value:
    print("Start date is after end date, please correct your date selection")
elif len(selected_ids) < 2:
    print("Please select at least two countlines so origin/destination pairs can be formed.")
else:
    # convert local datetime to UTC datetime
    start_date_utc = str(pd.to_datetime(start_date_input.value).tz_localize(timezone.value).astimezone(pytz.utc))
    end_date_utc = str(pd.to_datetime(end_date_input.value).tz_localize(timezone.value).astimezone(pytz.utc))
    date_range = get_date_range(start_date_utc, end_date_utc)

    num_requests = len(pairs) * len(date_range)
    # ~290 requests/minute ceiling
    est_minutes = num_requests / 290

    printmd("**Check your selection:**")
    print("Dates:", start_date_input.value, "to", end_date_input.value)
    print("Timezone:", timezone.value)
    print("Time bucket:", time_bucket)
    print("Classes:", classes_summary)
    print("Countlines selected:", len(selected_ids))
    print("Origin -> destination pairs:", len(pairs))
    print("Days:", len(date_range))
    print("Total API requests:", num_requests)
    print("Estimated download time: ~%.1f minutes (at the rate limit)" % est_minutes)
    if num_requests > 5000:
        print("\n⚠️  That's a lot of requests. Consider a larger time bucket or fewer countlines.")

### Getting the data

We now query Classified Journey Times for every origin/destination pair, requesting one day at a time and running several requests concurrently while respecting the API rate limits (300/minute, 7200/hour). The progress bar shows how far through we are.

In [ ]:
#@title  {vertical-output: true, display-mode: "form" }
#@markdown **Code cell:** Run this to get data from the API (can take a while)

JOURNEY_TIMES_URL = f'{api_url_base}/countline/journey_times'
MAX_WORKERS = 12
limiter = RateLimiter(max_per_min=290, max_per_hour=7000)

def fetch_pair_day(task):
    """Fetch one origin/destination pair for one day, with retry/backoff on errors."""
    origin, dest, frm, to = task
    params = {
        'origin_countline_id': origin,
        'destination_countline_id': dest,
        'origin_countline_direction': 'both',
        'destination_countline_direction': 'both',
        'from': frm,
        'to': to,
        'time_bucket': time_bucket,
    }
    # Aggregated => let the API return the combined total only.
    # Otherwise disaggregate, optionally filtered to the chosen classes.
    if not aggregated:
        params['disaggregate_classes'] = 'true'
        if request_classes:
            params['classes'] = request_classes
    for attempt in range(6):
        limiter.acquire()
        try:
            response = requests.get(JOURNEY_TIMES_URL, params=params, headers=headers, timeout=60)
        except requests.exceptions.RequestException:
            time.sleep(min(60, 2 ** attempt))
            continue
        if response.status_code == 200:
            return response.json()
        if response.status_code == 204:
            return {}
        if response.status_code == 429:
            time.sleep(min(60, 2 ** attempt))
            continue
        # other errors: don't hammer, give up on this task
        return {}
    return {}

# Build the task list: every pair x every day
tasks = [(o, d, frm, to) for (o, d) in pairs for (frm, to) in date_range]

rows = []
errors = 0
print(f"Requesting data for {len(pairs)} pairs over {len(date_range)} day(s) "
      f"= {len(tasks)} requests ...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(fetch_pair_day, t) for t in tasks]
    for future in tqdm(as_completed(futures), total=len(futures), desc="Collecting data"):
        try:
            result = future.result()
        except Exception:
            errors += 1
            continue
        for origin_id, dests in result.items():
            for dest_id, buckets in dests.items():
                for bucket in buckets:
                    journey_times = bucket.get("journey_times", {})
                    for cls, vals in journey_times.items():
                        rows.append({
                            "origin_id": origin_id,
                            "dest_id": dest_id,
                            "utc_from": bucket["from"],
                            "class": cls,
                            "journeys": vals.get("number_of_journeys"),
                            "p25": vals.get("25_percentile_journey_time"),
                            "mean": vals.get("mean_journey_time"),
                            "median": vals.get("median_journey_time"),
                            "p75": vals.get("75_percentile_journey_time"),
                        })

df_raw = pd.DataFrame(rows)
print(f"\nDone. Collected {len(df_raw)} rows.")
if errors:
    print(f"{errors} request(s) failed and were skipped.")
if len(df_raw) == 0:
    print("Note: No journey time data returned for the selected countlines, dates and classes.")

## Stage 3: Data Processing
Now we convert timestamps to your chosen timezone, attach readable countline labels and arrange the columns for export.

In [ ]:
#@title  {vertical-output: true, display-mode: "form" }
#@markdown **Code cell:** Run this to format the data for export

if len(df_raw) == 0:
    print("No data to process. Please run a different request.")
else:
    export = df_raw.copy()
    export["UTC Datetime"] = pd.to_datetime(export["utc_from"])
    export["Local Datetime"] = export["UTC Datetime"].dt.tz_convert(timezone.value)
    export["Origin Countline"] = export["origin_id"].map(label_map).fillna(export["origin_id"])
    export["Destination Countline"] = export["dest_id"].map(label_map).fillna(export["dest_id"])

    export = export.rename(columns={
        "class": "Class",
        "journeys": "Journeys",
        "p25": "25 Percentile",
        "mean": "Mean Journey Time",
        "median": "Median Journey Time",
        "p75": "75 Percentile",
    })

    column_order = [
        "UTC Datetime", "Local Datetime", "Origin Countline", "Destination Countline",
        "Class", "Journeys", "25 Percentile", "Mean Journey Time", "Median Journey Time",
        "75 Percentile",
    ]
    export = export[column_order].sort_values(
        ["UTC Datetime", "Origin Countline", "Destination Countline", "Class"]
    ).reset_index(drop=True)

    print(f"Preview of export ({len(export)} rows):")
    display(export.head(50))

## Stage 4: Data Export
Now let's write this to a .csv file. You can either save the file locally (it will show in your Downloads folder) or save it to a Google Drive.

- **Local Downloads Folder:** This might not work if your browser or computer is blocking downloads.
- **Google Drive:** If you want to save it in Google Drive, you will be asked for permission to connect to your Google Account.

In [ ]:
#@title  { vertical-output: true, display-mode: "form" }
#@markdown Select where to save the csv file
download_location = "Local folder" #@param [ "Local folder", "Google Drive"]
#@markdown Name your file
filename = "classified-journey-times" #@param {type:"string"}
#@markdown Hit run (▶)

if download_location == "Local folder":
    from google.colab import files
    export.to_csv(filename + ".csv", index=False)
    files.download(filename + ".csv")
else:
    from google.colab import drive
    drive.mount('/content/drive')
    path = '/content/drive/My Drive/'
    export.to_csv(path + filename + ".csv", index=False)